# Friends Demo — Phase 3: The Read Path

query embedding → similarity search → ranking → injection

We can't see raw embedding vectors (MemoryClient is hosted), but we can see and control the
real levers: `top_k`, `threshold`, `rerank`, and each result's `score`.

Same Maya/Jordan/Sam data, continued from Phase 2.


## 0. Reconnect, and turn decay off

Decay reranks by recency at search time. We turn it off so this notebook is a clean test of
relevance ranking alone -- we'll turn it back on deliberately in Phase 4.


In [ ]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))
client.project.update(decay=False)

existing = client.get_all(filters={"user_id": "maya"})
print(f"{len(existing.get('results', []))} memories on file for Maya")

## 1. Scores, not just text

In [ ]:
query = "what hobby is Maya doing these days?"

results = client.search(query=query, filters={"user_id": "maya"})
for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

## 2. Threshold experiment

In [ ]:
for threshold in [0.0, 0.3, 0.6, 0.9]:
    results = client.search(query=query, filters={"user_id": "maya"}, threshold=threshold)
    print(f"threshold={threshold} -> {len(results.get('results', []))} results")

In [ ]:
results = client.search(query=query, filters={"user_id": "maya"}, threshold=0.5)
for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

## 3. top_k experiment

In [ ]:
for k in [1, 3, 10]:
    results = client.search(query=query, filters={"user_id": "maya"}, top_k=k)
    print(f"top_k={k}:")
    for r in results.get("results", []):
        print(f"   {r['score']:.3f}  {r['memory']}")
    print()

## 4. Ranking, using the pottery/painting contradiction

Phase 2 found that adding "I switched to painting" doesn't necessarily delete the old
pottery fact. This is the read-path test of whether ranking compensates for that.


In [ ]:
results = client.search(
    query="what hobby is Maya doing these days?",
    filters={"user_id": "maya"},
    top_k=10,
)

for r in results.get("results", []):
    marker = ""
    if "painting" in r["memory"].lower():
        marker = "  <-- painting (new)"
    elif "pottery" in r["memory"].lower():
        marker = "  <-- pottery (stale)"
    print(f"{r['score']:.3f}  {r['memory']}{marker}")

If painting consistently outranks pottery, that's evidence the conflict gets resolved at
**read time** via scoring, not at write time via deletion. If they're close together, or
pottery ranks higher, that's a real limitation worth flagging, not smoothing over.


## 5. Bonus: rerank

In [ ]:
plain = client.search(query=query, filters={"user_id": "maya"}, top_k=5)
reranked = client.search(query=query, filters={"user_id": "maya"}, top_k=5, rerank=True)

print("rerank=False:")
for r in plain.get("results", []):
    print(f"   {r['score']:.3f}  {r['memory']}")

print("\nrerank=True:")
for r in reranked.get("results", []):
    print(f"   {r['score']:.3f}  {r['memory']}")

## 6. Injection comparison: does what you retrieve change what gets said?

In [ ]:
from lab_llm_config import complete

def answer_with_settings(question, user_id, **search_kwargs):
    results = client.search(query=question, filters={"user_id": user_id}, **search_kwargs)
    memories = [r["memory"] for r in results.get("results", [])]
    memory_block = "\n".join(f"- {m}" for m in memories)
    prompt = (
        "You're chatting with a friend. Use these facts if relevant, answer naturally "
        f"without mentioning 'stored memories':\n\n{memory_block}\n\nQuestion: {question}"
    )
    return complete(prompt), memories

In [ ]:
question = "what hobby is Maya doing these days?"

settings = {
    "top_1": {"top_k": 1},
    "top_5": {"top_k": 5},
    "threshold_0.5": {"threshold": 0.5, "top_k": 10},
}

for label, kwargs in settings.items():
    answer, memories = answer_with_settings(question, user_id="maya", **kwargs)
    print(f"--- {label} ({len(memories)} memories used) ---")
    print(answer)
    print()

## Wrap-up

1. Relevance is a number (`score`), and `threshold`/`top_k` are the two levers that turn it
   into an actual result set.
2. Section 4 is the direct test of whether ranking, not storage, resolves the pottery/painting
   contradiction.
3. Section 6 shows the part that's actually visible to a user: different retrieval settings
   produce different answers, not just different printed lists.

**Next up:** Phase 4 -- decay and reinforcement.
